# SPAD — Master End-to-End REDESIGNED Production Workflow

**Screening & Predictive Anomaly Detection for High-Reliability Electronic Components**

This production workflow implements the proposed SPAD architecture:

```text
NASA MOSFET transient MAT files
        |
        v
Frozen ON-window detection + scalar RDS extraction
        |
        v
4-stage RDS trajectory (0 / 33.33 / 66.67 / 100%)
        |
        +--> Module A: Progressive trajectory anomaly detection
        |       Stage 0:   [RDS0]
        |       Stage 33:  [RDS0, RDS33, Delta_0_33]
        |       Stage 66:  [RDS0, RDS33, RDS66, Delta_0_33, Delta_33_66]
        |       Stage 100: [RDS0, RDS33, RDS66, RDS100, all deltas]
        |       Output: stage-wise raw IF score + model flag
        |
        +--> Module B: Frozen future RDS prediction
        |       [RDS0, RDS33, ID33/ID0] -> Predicted_RDS100
        |
        +--> Module C: Engineer-defined hard limit
                scan ALL valid ON-state RDS(t) samples
                any RDS(t) > engineer limit -> LIMIT FLAG
        |
        v
Final SPAD result table
        |
        +--> optional Node/Express dry-run
```

### Important boundary

The old 2-feature Module A from `SPAD_V2_Final_Model_Package.pkl` is **not used** as the production Module A in this redesigned notebook. The new Module A is progressive and must be fitted/validated as a new model. The existing `.pkl` is used only for the frozen Module B regressor.

FFT/spectral experimentation remains excluded.

### V4 audit note

The notebook fails loudly on schema/selection mismatches. In particular, if the frozen run-level dataset says a run contains target-condition transients but Module C selects zero from the raw MAT file, the run is marked **ERROR** rather than silently treated as NOT_EVALUATED.


In [1]:
# ============================================================
# 0. IMPORTS + CONFIGURATION + ENGINEER LIMIT INPUT
# ============================================================

import os
import pickle
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt

from scipy.ndimage import median_filter
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")

DOWNLOADS = Path(r"C:\Users\Admin\Downloads")
RAW_DATA_ROOT = Path(r"D:\MOSFET_Thermal_Overstress_Aging_v0")

MASTER_RUN_FILE = DOWNLOADS / "MOSFET_199_200C_RDSon_RunLevel.csv"
RUNLEVEL_FILE = DOWNLOADS / "MOSFET_199_200C_V2_Electrical_Features_RunLevel.csv"
CHECKPOINT_FILE = DOWNLOADS / "MOSFET_Normalized_4Checkpoints.csv"
MODEL_PACKAGE_FILE = DOWNLOADS / "SPAD_V2_Final_Model_Package.pkl"
RESULT_FILE = DOWNLOADS / "SPAD_V4_End_to_End_Inference.csv"

TARGET_CONDITION = (199, 200, 10, 5, 1000, 40)

# Entire component trajectories are held out of Module A training for audit/benchmarking.
HELD_OUT_IDS = [10, 13]

REBUILD_RUNLEVEL = False
REBUILD_CHECKPOINTS = False

# Production delivery is disabled by default.
# Set True only after the Node/Express dry-run contract is verified.
SEND_TO_BACKEND = True
BACKEND_BASE_URL = os.getenv(
    "SPAD_BACKEND_URL",
    "https://sih26-spad.onrender.com"
)
LOT_ID = os.getenv("SPAD_LOT_ID", "").strip()

# Engineering configuration
# 8.0 Ω is the current configured engineering hard limit.
# Change only through the engineer-approved configuration.
_engineering_limit_raw = os.getenv("SPAD_RDS_LIMIT_OHM", "8.0")

# Provisional analytical minimum-current validity gate for instantaneous RDS(t).
# This is NOT an authoritative engineering specification.
PROVISIONAL_MIN_CURRENT_A = float(
    os.getenv("SPAD_MIN_VALID_CURRENT_A", "0.15")
)

if not np.isfinite(PROVISIONAL_MIN_CURRENT_A) or PROVISIONAL_MIN_CURRENT_A <= 0:
    raise ValueError(
        "Provisional minimum valid current must be a finite positive value."
    )

MODULE_A_N_ESTIMATORS = 500
MODULE_A_RANDOM_STATE = 42
MODULE_A_CONTAMINATION = "auto"

# Engineer-defined hard limit. Never invented by the algorithm.
try:
    ENGINEERING_LIMIT_OHM = float(_engineering_limit_raw)
except (TypeError, ValueError) as exc:
    raise ValueError("Engineering RDS hard limit must be numeric (Ω).") from exc

if not np.isfinite(ENGINEERING_LIMIT_OHM) or ENGINEERING_LIMIT_OHM <= 0:
    raise ValueError("Engineering RDS hard limit must be a finite positive value.")

print("=" * 100)
print("SPAD V4 — PROGRESSIVE ML + ENGINEER HARD-LIMIT SYSTEM")
print("=" * 100)
print("Raw data root        :", RAW_DATA_ROOT)
print("Run-level file       :", RUNLEVEL_FILE)
print("Checkpoint file      :", CHECKPOINT_FILE)
print("Module B package     :", MODEL_PACKAGE_FILE)
print("Engineer RDS limit   :", ENGINEERING_LIMIT_OHM, "Ω")
print("Min valid ID for Module C:", PROVISIONAL_MIN_CURRENT_A, "A")
print("Backend send enabled :", SEND_TO_BACKEND)

SPAD V4 — PROGRESSIVE ML + ENGINEER HARD-LIMIT SYSTEM
Raw data root        : D:\MOSFET_Thermal_Overstress_Aging_v0
Run-level file       : C:\Users\Admin\Downloads\MOSFET_199_200C_V2_Electrical_Features_RunLevel.csv
Checkpoint file      : C:\Users\Admin\Downloads\MOSFET_Normalized_4Checkpoints.csv
Module B package     : C:\Users\Admin\Downloads\SPAD_V2_Final_Model_Package.pkl
Engineer RDS limit   : 8.0 Ω
Min valid ID for Module C: 0.15 A
Backend send enabled : True


In [2]:
# ============================================================
# 1. ENGINEERING LIMIT VALIDATION
# ============================================================

def set_engineering_limit(limit_ohm):
    try:
        limit_ohm = float(limit_ohm)
    except (TypeError, ValueError) as exc:
        raise ValueError("Engineering RDS limit must be numeric (Ω).") from exc
    if not np.isfinite(limit_ohm) or limit_ohm <= 0:
        raise ValueError("Engineering RDS limit must be finite and > 0 Ω.")
    return limit_ohm

ENGINEERING_LIMIT_OHM = set_engineering_limit(ENGINEERING_LIMIT_OHM)
print(f"Engineering hard-limit detector configured: {ENGINEERING_LIMIT_OHM:.6f} Ω")


Engineering hard-limit detector configured: 8.000000 Ω


## 2. Frozen transient methodology

The scalar RDS feature remains unchanged:

1. Median-filter gate signal with kernel 5.
2. Estimate gate low/high from the 5th/95th percentiles.
3. Use ON = low + 70% excursion and OFF = low + 30% excursion.
4. Select the longest ON interval.
5. Scalar RDS = median(VDS/ID) over the 70–90% portion of that ON interval.
6. Module C uses the **entire valid ON interval** for the engineer-defined hard-limit check.


In [3]:
# ============================================================
# 3. TRANSIENT HELPERS + MODULE C SCANNING
# ============================================================

def get_pwm_condition(p):
    """Return the exact condition tuple used by the frozen run-level pipeline."""
    values = []
    fields = (
        "lowTemp", "highTemp", "gateVoltage",
        "supplyVoltage", "switchingFrequency", "dutyCycle"
    )
    for field in fields:
        value = getattr(p, field, np.nan)
        try:
            value = float(value)
        except Exception:
            value = np.nan
        if not np.isfinite(value):
            return None
        values.append(int(round(value)))
    return tuple(values)


def build_pwm_timeline(measurement):
    """Build sorted (timeEpoch, condition) records from the MATLAB PWM state."""
    if not hasattr(measurement, "pwmTempControllerState"):
        return np.array([], dtype=float), []

    pwm = np.atleast_1d(measurement.pwmTempControllerState)
    rows = []
    for p in pwm:
        try:
            t = float(getattr(p, "timeEpoch"))
            condition = get_pwm_condition(p)
            if np.isfinite(t) and condition is not None:
                rows.append((t, condition))
        except Exception:
            continue

    rows.sort(key=lambda x: x[0])
    if not rows:
        return np.array([], dtype=float), []
    return np.array([x[0] for x in rows], dtype=float), [x[1] for x in rows]


def detect_on_window(gate):
    """Frozen gate ON-window detector: median filter 5, 5/95 percentiles, 70/30 thresholds."""
    gate = np.asarray(gate, dtype=float).squeeze()
    valid = np.isfinite(gate)
    if valid.sum() < 10:
        return None

    gate_clean = gate.copy()
    gate_clean[~valid] = np.nanmedian(gate_clean[valid])
    gate_filtered = median_filter(gate_clean, size=5, mode="nearest")

    low = np.percentile(gate_filtered, 5)
    high = np.percentile(gate_filtered, 95)
    amplitude = high - low
    if amplitude <= 0:
        return None

    on_threshold = low + 0.70 * amplitude
    off_threshold = low + 0.30 * amplitude

    state = np.zeros(len(gate_filtered), dtype=bool)
    is_on = False
    for i, value in enumerate(gate_filtered):
        if not is_on and value >= on_threshold:
            is_on = True
        elif is_on and value <= off_threshold:
            is_on = False
        state[i] = is_on

    changes = np.diff(np.concatenate(([False], state, [False])).astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]
    regions = [(int(s), int(e)) for s, e in zip(starts, ends) if e > s]
    return max(regions, key=lambda x: x[1] - x[0]) if regions else None


def extract_transient_features(transient):
    """Frozen scalar RDS extraction over the 70–90% portion of the valid ON interval."""
    td = transient.timeDomain
    gate = np.asarray(td.gateSignalVoltage, dtype=float).squeeze()
    vds = np.asarray(td.drainSourceVoltage, dtype=float).squeeze()
    ids = np.asarray(td.drainCurrent, dtype=float).squeeze()

    n = min(len(gate), len(vds), len(ids))
    if n < 20:
        return None
    gate, vds, ids = gate[:n], vds[:n], ids[:n]

    detected = detect_on_window(gate)
    if detected is None:
        return None

    start, end = detected
    on_length = end - start
    if on_length < 20:
        return None

    a = start + int(0.70 * on_length)
    b = start + int(0.90 * on_length)
    if b <= a:
        return None

    v = vds[a:b]
    i = ids[a:b]
    valid = np.isfinite(v) & np.isfinite(i) & (np.abs(i) > 1e-6)
    if valid.sum() < 5:
        return None

    r = v[valid] / i[valid]
    v = v[valid]
    i = i[valid]
    valid_r = np.isfinite(r) & (r > 0)
    r, v, i = r[valid_r], v[valid_r], i[valid_r]
    if len(r) < 5:
        return None

    return {
        "VDS_median": float(np.median(v)),
        "ID_median": float(np.median(i)),
        "RDS_median": float(np.median(r)),
        "RDS_mean": float(np.mean(r)),
        "RDS_std": float(np.std(r)),
        "N_samples": int(len(r)),
    }


LIMIT_RESULT_KEYS = [
    "Engineering_Limit_Ohm",
    "Max_RDS_Instantaneous_Ohm",
    "Max_RDS_Transient_ID",
    "Max_RDS_Time_us",
    "Limit_Exceedance_Count",
    "First_Limit_Exceedance_Time_us",
    "First_Limit_Exceedance_Transient_ID",
    "Limit_Exceedance_Margin_Ohm",
    "Limit_Exceedance_Flag",
    "Valid_RDS_Sample_Count",
    "Target_Condition_Transient_Count",
    "Limit_Scan_Status",
    "Limit_Scan_Reason",
    "Raw_Transient_Count",
    "PWM_State_Count",
    "Target_Condition_Match_Attempt_Count",
    "Provisional_Min_Current_A",
]


def empty_limit_result(limit_ohm, status, reason="", raw_transient_count=0,
                       pwm_state_count=0, match_attempt_count=0):
    return {
        "Engineering_Limit_Ohm": float(limit_ohm) if limit_ohm is not None else np.nan,
        "Max_RDS_Instantaneous_Ohm": np.nan,
        "Max_RDS_Transient_ID": np.nan,
        "Max_RDS_Time_us": np.nan,
        "Limit_Exceedance_Count": 0,
        "First_Limit_Exceedance_Time_us": np.nan,
        "First_Limit_Exceedance_Transient_ID": np.nan,
        "Limit_Exceedance_Margin_Ohm": np.nan,
        "Limit_Exceedance_Flag": "NOT_EVALUATED",
        "Valid_RDS_Sample_Count": 0,
        "Target_Condition_Transient_Count": 0,
        "Limit_Scan_Status": status,
        "Limit_Scan_Reason": reason,
        "Raw_Transient_Count": int(raw_transient_count),
        "PWM_State_Count": int(pwm_state_count),
        "Target_Condition_Match_Attempt_Count": int(match_attempt_count),
        "Provisional_Min_Current_A": float(PROVISIONAL_MIN_CURRENT_A),
    }


def scan_transient_limit(transient, transient_id, dt, engineering_limit_ohm):
    """Scan every valid RDS(t) sample in the complete detected ON interval."""
    td = transient.timeDomain
    gate = np.asarray(td.gateSignalVoltage, dtype=float).squeeze()
    vds = np.asarray(td.drainSourceVoltage, dtype=float).squeeze()
    ids = np.asarray(td.drainCurrent, dtype=float).squeeze()

    n = min(len(gate), len(vds), len(ids))
    if n < 20:
        return None
    gate, vds, ids = gate[:n], vds[:n], ids[:n]

    detected = detect_on_window(gate)
    if detected is None:
        return None

    start, end = detected
    idx = np.arange(start, end)
    on_vds = vds[start:end]
    on_ids = ids[start:end]
    time_us = idx * float(dt) * 1e6

    # Physical/numerical validity:
    # finite signals plus the provisional minimum-current gate.
    valid = (
        np.isfinite(on_vds)
        & np.isfinite(on_ids)
        & (np.abs(on_ids) >= PROVISIONAL_MIN_CURRENT_A)
    )
    if valid.sum() == 0:
        return None

    rds = on_vds[valid] / on_ids[valid]
    t = time_us[valid]
    valid_rds = np.isfinite(rds) & (rds > 0)
    rds, t = rds[valid_rds], t[valid_rds]
    if len(rds) == 0:
        return None

    mask = rds > engineering_limit_ohm
    imax = int(np.argmax(rds))
    first = np.where(mask)[0][0] if mask.any() else None

    return {
        "Engineering_Limit_Ohm": float(engineering_limit_ohm),
        "Max_RDS_Instantaneous_Ohm": float(rds[imax]),
        "Max_RDS_Transient_ID": int(transient_id),
        "Max_RDS_Time_us": float(t[imax]),
        "Limit_Exceedance_Count": int(mask.sum()),
        "First_Limit_Exceedance_Time_us": float(t[first]) if first is not None else np.nan,
        "First_Limit_Exceedance_Transient_ID": int(transient_id) if first is not None else np.nan,
        "Limit_Exceedance_Margin_Ohm": float(np.max(rds) - engineering_limit_ohm),
        "Limit_Exceedance_Flag": "FLAGGED" if mask.any() else "NOT FLAGGED",
        "Valid_RDS_Sample_Count": int(len(rds)),
        "Target_Condition_Transient_Count": 1,
        "Limit_Scan_Status": "EVALUATED",
        "Limit_Scan_Reason": "",
    }


def scan_run_hard_limit(mat_file, engineering_limit_ohm):
    """Scan ALL target-condition transients in one raw MAT run."""
    data = scipy.io.loadmat(mat_file, struct_as_record=False, squeeze_me=True)
    if "measurement" not in data:
        return empty_limit_result(
            engineering_limit_ohm, "ERROR", "MAT file has no 'measurement' object."
        )

    measurement = data["measurement"]
    pwm_times, pwm_conditions = build_pwm_timeline(measurement)
    transients = np.atleast_1d(getattr(measurement, "transient", []))

    raw_count = len(transients)
    pwm_count = len(pwm_times)
    match_attempts = 0
    evaluated = []

    if raw_count == 0:
        return empty_limit_result(
            engineering_limit_ohm, "NOT_EVALUATED",
            "No transient records found in MAT file.", raw_count, pwm_count, match_attempts
        )

    if pwm_count == 0:
        return empty_limit_result(
            engineering_limit_ohm, "ERROR",
            "No usable PWM controller timeline; target-condition transient selection cannot be verified.",
            raw_count, pwm_count, match_attempts
        )

    for transient_id, tr in enumerate(transients):
        try:
            transient_time = float(getattr(tr, "timeEpoch"))
            dt = float(np.asarray(tr.timeDomain.dt).squeeze())
        except Exception:
            continue

        if not np.isfinite(transient_time) or not np.isfinite(dt) or dt <= 0:
            continue

        # Use the last PWM state at or immediately before the transient time,
        # exactly matching the frozen run-level selection logic.
        pwm_idx = np.searchsorted(pwm_times, transient_time, side="right") - 1
        if pwm_idx < 0:
            continue

        match_attempts += 1
        if pwm_conditions[pwm_idx] != TARGET_CONDITION:
            continue

        rec = scan_transient_limit(tr, transient_id, dt, engineering_limit_ohm)
        if rec is not None:
            evaluated.append(rec)

    if not evaluated:
        reason = (
            f"No valid target-condition transient samples evaluated. "
            f"raw_transients={raw_count}, pwm_states={pwm_count}, "
            f"condition_match_attempts={match_attempts}, target_condition={TARGET_CONDITION}."
        )
        return empty_limit_result(
            engineering_limit_ohm, "NOT_EVALUATED", reason,
            raw_count, pwm_count, match_attempts
        )

    max_rec = max(evaluated, key=lambda x: x["Max_RDS_Instantaneous_Ohm"])
    flagged = [x for x in evaluated if x["Limit_Exceedance_Flag"] == "FLAGGED"]
    first_rec = min(
        flagged,
        key=lambda x: x["First_Limit_Exceedance_Time_us"]
    ) if flagged else None

    return {
        "Engineering_Limit_Ohm": float(engineering_limit_ohm),
        "Max_RDS_Instantaneous_Ohm": float(max_rec["Max_RDS_Instantaneous_Ohm"]),
        "Max_RDS_Transient_ID": int(max_rec["Max_RDS_Transient_ID"]),
        "Max_RDS_Time_us": float(max_rec["Max_RDS_Time_us"]),
        "Limit_Exceedance_Count": int(sum(x["Limit_Exceedance_Count"] for x in evaluated)),
        "First_Limit_Exceedance_Time_us": float(first_rec["First_Limit_Exceedance_Time_us"]) if first_rec else np.nan,
        "First_Limit_Exceedance_Transient_ID": int(first_rec["First_Limit_Exceedance_Transient_ID"]) if first_rec else np.nan,
        "Limit_Exceedance_Margin_Ohm": float(max_rec["Max_RDS_Instantaneous_Ohm"] - engineering_limit_ohm),
        "Limit_Exceedance_Flag": "FLAGGED" if flagged else "NOT FLAGGED",
        "Valid_RDS_Sample_Count": int(sum(x["Valid_RDS_Sample_Count"] for x in evaluated)),
        "Target_Condition_Transient_Count": int(len(evaluated)),
        "Limit_Scan_Status": "EVALUATED",
        "Limit_Scan_Reason": "",
        "Raw_Transient_Count": int(raw_count),
        "PWM_State_Count": int(pwm_count),
        "Target_Condition_Match_Attempt_Count": int(match_attempts),
        "Provisional_Min_Current_A": float(PROVISIONAL_MIN_CURRENT_A),
    }


In [4]:
# ============================================================
# 4. PROCESS ONE RUN — frozen scalar RDS extraction only
# ============================================================

def process_run(test_id, run_id):
    mat_file = RAW_DATA_ROOT / f"Test_{test_id}_run_{run_id}.mat"
    if not mat_file.exists():
        raise FileNotFoundError(mat_file)
    data = scipy.io.loadmat(mat_file, struct_as_record=False, squeeze_me=True)
    measurement = data["measurement"]
    pwm_times, pwm_conditions = build_pwm_timeline(measurement)
    transients = np.atleast_1d(measurement.transient)
    rows = []
    for transient_id, transient in enumerate(transients):
        try:
            transient_time = float(getattr(transient, "timeEpoch"))
        except Exception:
            continue
        if not np.isfinite(transient_time):
            continue
        pwm_idx = np.searchsorted(pwm_times, transient_time, side="right") - 1
        if pwm_idx < 0 or pwm_conditions[pwm_idx] != TARGET_CONDITION:
            continue
        features = extract_transient_features(transient)
        if features is None:
            continue
        rows.append({"Test_ID": int(test_id), "Run_ID": int(run_id), "Transient_ID": int(transient_id), **features})
    return pd.DataFrame(rows)


In [5]:
# ============================================================
# 5. BUILD / LOAD RUN-LEVEL ELECTRICAL DATA
# ============================================================

if REBUILD_RUNLEVEL:
    run_index = pd.read_csv(MASTER_RUN_FILE)
    all_runs = []
    for _, row in run_index.iterrows():
        test_id, run_id = int(row["Test_ID"]), int(row["Run_ID"])
        try:
            df_run = process_run(test_id, run_id)
        except Exception as exc:
            print(f"ERROR Test {test_id} Run {run_id}: {exc}")
            continue
        if df_run.empty:
            continue
        all_runs.append({
            "Test_ID": test_id,
            "Run_ID": run_id,
            "Cumulative_Aging_Hours": float(row["Cumulative_Aging_Hours"]),
            "Matched_Transient_Count": len(df_run),
            "RDSon_Median_Ohm": df_run["RDS_median"].median(),
            "RDSon_Mean_Ohm": df_run["RDS_median"].mean(),
            "RDSon_STD_Ohm": df_run["RDS_median"].std(),
            "ID_ON_Median_A": df_run["ID_median"].median(),
            "ID_ON_Mean_A": df_run["ID_median"].mean(),
            "ID_ON_STD_A": df_run["ID_median"].std(),
            "VDS_ON_Median_V": df_run["VDS_median"].median(),
            "VDS_ON_Mean_V": df_run["VDS_median"].mean(),
            "VDS_ON_STD_V": df_run["VDS_median"].std(),
        })
    runlevel = pd.DataFrame(all_runs).sort_values(["Test_ID", "Cumulative_Aging_Hours"]).reset_index(drop=True)
    runlevel.to_csv(RUNLEVEL_FILE, index=False)
else:
    if not RUNLEVEL_FILE.exists():
        raise FileNotFoundError(f"Run-level file not found: {RUNLEVEL_FILE}")
    runlevel = pd.read_csv(RUNLEVEL_FILE)

print("Run-level shape:", runlevel.shape)
print("Physical Test IDs:", runlevel["Test_ID"].nunique())


Run-level shape: (67, 13)
Physical Test IDs: 15


In [6]:
# ============================================================
# 6. BUILD / LOAD NORMALIZED 4-CHECKPOINT DATASET
# ============================================================

CHECKPOINTS = [
    (1, 0.000000, "0h_equivalent"),
    (2, 0.333333, "24h_equivalent"),
    (3, 0.666667, "96h_equivalent"),
    (4, 1.000000, "168h_equivalent"),
]

def build_checkpoint_values(group):
    group = group.sort_values("Cumulative_Aging_Hours").reset_index(drop=True)
    ages = group["Cumulative_Aging_Hours"].to_numpy(dtype=float)
    rds = group["RDSon_Median_Ohm"].to_numpy(dtype=float)
    test_id = int(group["Test_ID"].iloc[0])
    if len(group) < 2:
        return []
    age_min, age_max = ages.min(), ages.max()
    span = age_max - age_min
    if span <= 0:
        return []
    rows = []
    for checkpoint_id, progress, label in CHECKPOINTS:
        target_age = age_min + progress * span
        rows.append({
            "Test_ID": test_id,
            "Checkpoint": checkpoint_id,
            "Normalized_Progress": progress,
            "Challenge_Equivalent_Label": label,
            "Observed_Age_Min_Hours": age_min,
            "Observed_Age_Max_Hours": age_max,
            "Checkpoint_Age_Equivalent_Hours": target_age,
            "RDSon_Interpolated_Ohm": np.interp(target_age, ages, rds),
            "Nearest_Actual_Run_ID": int(group.loc[int(np.argmin(np.abs(ages-target_age))), "Run_ID"]),
            "Nearest_Actual_Age_Hours": ages[int(np.argmin(np.abs(ages-target_age)))],
            "Number_of_Actual_Runs": len(group),
        })
    return rows

if REBUILD_CHECKPOINTS:
    rows = []
    for _, group in runlevel.groupby("Test_ID"):
        rows.extend(build_checkpoint_values(group))
    checkpoints_df = pd.DataFrame(rows)
    checkpoints_df.to_csv(CHECKPOINT_FILE, index=False)
else:
    if not CHECKPOINT_FILE.exists():
        raise FileNotFoundError(f"Checkpoint file not found: {CHECKPOINT_FILE}")
    checkpoints_df = pd.read_csv(CHECKPOINT_FILE)

print("Checkpoint dataset shape:", checkpoints_df.shape)


Checkpoint dataset shape: (60, 11)


In [7]:
# ============================================================
# 7. ALIGN ID FEATURES TO CHECKPOINT AGES + BUILD WIDE TRAJECTORY
# ============================================================

records = []
for test_id, cp_group in checkpoints_df.groupby("Test_ID"):
    runs = runlevel[runlevel["Test_ID"] == test_id].sort_values("Cumulative_Aging_Hours")
    if len(runs) < 2:
        continue
    age = runs["Cumulative_Aging_Hours"].to_numpy(dtype=float)
    ids = runs["ID_ON_Median_A"].to_numpy(dtype=float)
    for _, cp in cp_group.iterrows():
        target_age = float(cp["Checkpoint_Age_Equivalent_Hours"])
        records.append({
            "Test_ID": int(test_id),
            "Checkpoint": int(cp["Checkpoint"]),
            "Normalized_Progress": float(cp["Normalized_Progress"]),
            "Checkpoint_Age_Equivalent_Hours": target_age,
            "RDSon_Ohm": float(cp["RDSon_Interpolated_Ohm"]),
            "ID_ON_A": float(np.interp(target_age, age, ids)),
        })

aligned = pd.DataFrame(records).sort_values(["Test_ID", "Checkpoint"]).reset_index(drop=True)

wide = aligned.pivot(index="Test_ID", columns="Checkpoint", values=["RDSon_Ohm", "ID_ON_A"])
wide.columns = [f"{feature}_{checkpoint}" for feature, checkpoint in wide.columns]
wide = wide.reset_index().rename(columns={
    "RDSon_Ohm_1":"RDS0", "RDSon_Ohm_2":"RDS33", "RDSon_Ohm_3":"RDS66", "RDSon_Ohm_4":"RDS100",
    "ID_ON_A_1":"ID0", "ID_ON_A_2":"ID33"
})

wide["Delta_RDS_0_33"] = wide["RDS33"] - wide["RDS0"]
wide["Delta_RDS_33_66"] = wide["RDS66"] - wide["RDS33"]
wide["Delta_RDS_66_100"] = wide["RDS100"] - wide["RDS66"]
wide["ID33_ID0_Ratio"] = wide["ID33"] / wide["ID0"].replace(0, np.nan)
wide["Delta_RDS"] = wide["Delta_RDS_0_33"]  # backward-compatible Module B field

required = ["Test_ID","RDS0","RDS33","RDS66","RDS100","ID0","ID33","Delta_RDS_0_33","Delta_RDS_33_66","Delta_RDS_66_100","ID33_ID0_Ratio"]
missing = [c for c in required if c not in wide.columns]
if missing:
    raise ValueError(f"Missing required trajectory fields: {missing}")

print("Wide trajectory shape:", wide.shape)
display(wide[required])


Wide trajectory shape: (15, 14)


,Test_ID,RDS0,RDS33,RDS66,RDS100,ID0,ID33,Delta_RDS_0_33,Delta_RDS_33_66,Delta_RDS_66_100,ID33_ID0_Ratio
0,6,0.483105,0.589925,0.696745,0.803565,7.645391,6.879959,0.106820,0.106820,0.106820,0.899883
1,8,0.486021,0.528422,0.630160,0.660393,7.620788,7.249303,0.042401,0.101739,0.030233,0.951254
2,9,0.512634,0.518506,0.629173,0.635507,7.883223,7.827864,0.005872,0.110667,0.006334,0.992978
3,10,13.334364,13.612318,13.235870,14.374252,0.327429,0.297566,0.277954,-0.376448,1.138382,0.908793
4,11,0.518307,0.508018,0.609227,0.603363,7.743804,7.704160,-0.010290,0.101210,-0.005864,0.994881
5,12,0.510677,0.510597,0.636908,0.610565,7.678196,7.723910,-0.000080,0.126311,-0.026342,1.005954
6,13,0.513423,0.544736,0.568940,24.675459,7.612587,7.339782,0.031312,0.024204,24.106520,0.964164
7,14,0.513944,0.542102,0.626834,0.672058,7.587984,7.383834,0.028157,0.084733,0.045224,0.973096
8,20,0.572615,0.605515,0.638415,0.671315,7.202533,6.945567,0.032900,0.032900,0.032900,0.964323
9,23,0.563583,0.607752,0.651922,0.696091,7.321449,6.956502,0.044170,0.044170,0.044170,0.950154


In [8]:
# ============================================================
# 8. LOAD FROZEN MODEL PACKAGE — MODULE B ONLY
# ============================================================

with open(MODEL_PACKAGE_FILE, "rb") as f:
    model_package = pickle.load(f)

module_b = model_package["module_b"]
B_model = module_b["model"]
B_FEATURES = module_b["features"]

assert B_FEATURES == ["RDS0", "RDS33", "ID33_ID0_Ratio"]

print("Frozen Module B verified:")
print("  Model:", module_b["model_type"])
print("  Features:", B_FEATURES)
print("  Parameters:", module_b["parameters"])
print("\nLegacy Module A inside the old .pkl is intentionally NOT used in this redesigned system.")


Frozen Module B verified:
  Model: RandomForestRegressor
  Features: ['RDS0', 'RDS33', 'ID33_ID0_Ratio']
  Parameters: {'n_estimators': 100, 'max_depth': 2, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 1.0, 'random_state': 42}

Legacy Module A inside the old .pkl is intentionally NOT used in this redesigned system.


## 9. Module A — Progressive trajectory anomaly detection

Four stage-specific Isolation Forests are fitted only on the non-held-out component IDs. Each stage uses only information available at that stage; no future checkpoint is used.


In [9]:
# ============================================================
# 9. MODULE A — FIT PROGRESSIVE TRAJECTORY DETECTORS
# ============================================================

STAGE_FEATURES = {
    0: ["RDS0"],
    33: ["RDS0", "RDS33", "Delta_RDS_0_33"],
    66: ["RDS0", "RDS33", "RDS66", "Delta_RDS_0_33", "Delta_RDS_33_66"],
    100: ["RDS0", "RDS33", "RDS66", "RDS100", "Delta_RDS_0_33", "Delta_RDS_33_66", "Delta_RDS_66_100"],
}

normal_mask = ~wide["Test_ID"].isin(HELD_OUT_IDS)
progressive_models = {}

for stage, features in STAGE_FEATURES.items():
    train = wide.loc[normal_mask, features].to_numpy(dtype=float)
    if not np.isfinite(train).all():
        raise ValueError(f"Non-finite Module A training input at stage {stage}%.")
    if len(train) < 2:
        raise ValueError(f"Not enough normal components to train stage {stage}% detector.")
    model = IsolationForest(
        n_estimators=MODULE_A_N_ESTIMATORS,
        max_samples=min(13, len(train)),
        contamination=MODULE_A_CONTAMINATION,
        random_state=MODULE_A_RANDOM_STATE,
    )
    model.fit(train)
    progressive_models[stage] = model
    print(f"Stage {stage}%: trained on {len(train)} normal components, features={features}")


Stage 0%: trained on 13 normal components, features=['RDS0']
Stage 33%: trained on 13 normal components, features=['RDS0', 'RDS33', 'Delta_RDS_0_33']
Stage 66%: trained on 13 normal components, features=['RDS0', 'RDS33', 'RDS66', 'Delta_RDS_0_33', 'Delta_RDS_33_66']
Stage 100%: trained on 13 normal components, features=['RDS0', 'RDS33', 'RDS66', 'RDS100', 'Delta_RDS_0_33', 'Delta_RDS_33_66', 'Delta_RDS_66_100']


In [10]:
# ============================================================
# 10. MODULE A — APPLY PROGRESSIVE DETECTORS
# ============================================================

for stage, features in STAGE_FEATURES.items():
    model = progressive_models[stage]
    X = wide[features].to_numpy(dtype=float)
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite Module A inference input at stage {stage}%.")

    raw_score = model.decision_function(X)
    model_flag = model.predict(X) == -1

    train_scores = model.decision_function(
        wide.loc[normal_mask, features].to_numpy(dtype=float)
    )
    novelty = np.array([
        100.0 * np.mean(train_scores >= s) for s in raw_score
    ])

    wide[f"Module_A_IF_Score_{stage}"] = raw_score
    wide[f"Module_A_Novelty_Percentile_{stage}"] = novelty
    wide[f"Module_A_Flag_{stage}"] = np.where(model_flag, "FLAGGED", "NOT FLAGGED")

stage_flag_cols = [f"Module_A_Flag_{s}" for s in STAGE_FEATURES]
wide["Progressive_ML_Flag"] = np.where(
    wide[stage_flag_cols].eq("FLAGGED").any(axis=1),
    "FLAGGED",
    "NOT FLAGGED"
)

def first_flag_stage(row):
    for stage in [0, 33, 66, 100]:
        if row[f"Module_A_Flag_{stage}"] == "FLAGGED":
            return stage
    return np.nan

wide["First_Anomaly_Stage"] = wide.apply(first_flag_stage, axis=1)

display(wide[[
    "Test_ID",
    "Module_A_IF_Score_0", "Module_A_IF_Score_33", "Module_A_IF_Score_66", "Module_A_IF_Score_100",
    "Module_A_Flag_0", "Module_A_Flag_33", "Module_A_Flag_66", "Module_A_Flag_100",
    "Progressive_ML_Flag", "First_Anomaly_Stage"
]])


,Test_ID,Module_A_IF_Score_0,Module_A_IF_Score_33,Module_A_IF_Score_66,Module_A_IF_Score_100,Module_A_Flag_0,Module_A_Flag_33,Module_A_Flag_66,Module_A_Flag_100,Progressive_ML_Flag,First_Anomaly_Stage
0,6,-0.095767,-0.164192,-0.130023,-0.160149,FLAGGED,FLAGGED,FLAGGED,FLAGGED,FLAGGED,0.0
1,8,-0.067558,-0.036578,0.003662,0.028162,FLAGGED,FLAGGED,NOT FLAGGED,NOT FLAGGED,FLAGGED,0.0
2,9,0.061678,0.055694,0.061012,0.062381,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NaN
3,10,-0.030064,-0.143575,-0.115911,-0.146008,FLAGGED,FLAGGED,FLAGGED,FLAGGED,FLAGGED,0.0
4,11,-0.021844,-0.012515,0.023918,0.021784,FLAGGED,FLAGGED,NOT FLAGGED,NOT FLAGGED,FLAGGED,0.0
5,12,0.038035,0.056530,0.017437,0.000610,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NaN
6,13,0.055620,0.013196,-0.009126,-0.068796,NOT FLAGGED,NOT FLAGGED,FLAGGED,FLAGGED,FLAGGED,66.0
7,14,0.053862,0.017112,0.039582,0.046907,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED,NaN
8,20,-0.011346,-0.025285,-0.002627,0.026834,FLAGGED,FLAGGED,FLAGGED,NOT FLAGGED,FLAGGED,0.0
9,23,-0.036992,-0.046384,-0.027536,-0.005588,FLAGGED,FLAGGED,FLAGGED,FLAGGED,FLAGGED,0.0


In [11]:
# ============================================================
# 11. MODULE B — FROZEN FUTURE RDS100 PREDICTION
# ============================================================

X_B = wide[B_FEATURES].to_numpy(dtype=float)
if not np.isfinite(X_B).all():
    raise ValueError("Module B input contains non-finite values.")
wide["Predicted_RDS100"] = B_model.predict(X_B)

print("Module B predictions computed from the frozen regressor.")
display(wide[["Test_ID", "RDS0", "RDS33", "ID33_ID0_Ratio", "Predicted_RDS100"]])


Module B predictions computed from the frozen regressor.


,Test_ID,RDS0,RDS33,ID33_ID0_Ratio,Predicted_RDS100
0,6,0.483105,0.589925,0.899883,0.691441
1,8,0.486021,0.528422,0.951254,0.680075
2,9,0.512634,0.518506,0.992978,0.628772
3,10,13.334364,13.612318,0.908793,0.691119
4,11,0.518307,0.508018,0.994881,0.580966
5,12,0.510677,0.510597,1.005954,0.578886
6,13,0.513423,0.544736,0.964164,0.680075
7,14,0.513944,0.542102,0.973096,0.675940
8,20,0.572615,0.605515,0.964323,0.691119
9,23,0.563583,0.607752,0.950154,0.691119


## 12. Module C — Engineer-defined instantaneous hard-limit detector

This detector is deterministic and independent of ML. It scans **all target-condition transients and every valid ON-state RDS(t) sample**. If any valid sample exceeds the engineer-supplied limit, that run is flagged. A component/Test_ID is limit-flagged if any of its evaluated runs is limit-flagged.


In [12]:
# ============================================================
# 12. MODULE C — SCAN ALL RUNS / ALL VALID ON-STATE SAMPLES
# ============================================================

limit_rows = []

required_run_cols = {"Test_ID", "Run_ID", "Matched_Transient_Count"}
missing_run_cols = sorted(required_run_cols - set(runlevel.columns))
if missing_run_cols:
    raise ValueError(
        "Run-level input is missing required Module C audit columns: "
        + ", ".join(missing_run_cols)
    )

for _, run in runlevel[["Test_ID", "Run_ID", "Matched_Transient_Count"]].drop_duplicates().iterrows():
    test_id = int(run["Test_ID"])
    run_id = int(run["Run_ID"])
    expected_target_transients = int(run["Matched_Transient_Count"])
    mat_file = RAW_DATA_ROOT / f"Test_{test_id}_run_{run_id}.mat"

    if not mat_file.exists():
        rec = empty_limit_result(
            ENGINEERING_LIMIT_OHM,
            "NOT_EVALUATED",
            f"MAT file not found: {mat_file}",
        )
    else:
        try:
            rec = scan_run_hard_limit(mat_file, ENGINEERING_LIMIT_OHM)

            # Hypervigilant consistency check: the frozen run-level pipeline
            # already knows whether target-condition transients existed.
            # If it expected >0 but Module C selected none, surface a real
            # selection mismatch rather than silently reporting NOT_EVALUATED.
            if expected_target_transients > 0 and rec["Target_Condition_Transient_Count"] == 0:
                rec["Limit_Scan_Status"] = "ERROR"
                rec["Limit_Scan_Reason"] = (
                    "Target-condition selection mismatch: "
                    f"run-level expected {expected_target_transients} matched transient(s), "
                    "but Module C selected 0. " + rec.get("Limit_Scan_Reason", "")
                ).strip()
        except Exception as exc:
            rec = empty_limit_result(
                ENGINEERING_LIMIT_OHM,
                "ERROR",
                f"Module C execution error: {type(exc).__name__}: {exc}",
            )

    limit_rows.append({
        "Test_ID": test_id,
        "Run_ID": run_id,
        "Expected_Target_Transient_Count": expected_target_transients,
        **rec,
    })

limit_runlevel = pd.DataFrame(limit_rows)

missing_limit_cols = [c for c in LIMIT_RESULT_KEYS if c not in limit_runlevel.columns]
if missing_limit_cols:
    raise RuntimeError(
        "Module C schema failure. Missing columns: " + ", ".join(missing_limit_cols)
    )

# Component-level result.
component_limit = (
    limit_runlevel.groupby("Test_ID", as_index=False)
    .agg(
        Engineering_Limit_Ohm=("Engineering_Limit_Ohm", "first"),
        Max_RDS_Instantaneous_Ohm=("Max_RDS_Instantaneous_Ohm", "max"),
        Limit_Exceedance_Count=("Limit_Exceedance_Count", "sum"),
        Limit_Exceedance_Runs=("Limit_Exceedance_Flag", lambda s: int((s == "FLAGGED").sum())),
        Runs_Evaluated=("Limit_Scan_Status", lambda s: int((s == "EVALUATED").sum())),
        Runs_With_Error=("Limit_Scan_Status", lambda s: int((s == "ERROR").sum())),
        Valid_RDS_Sample_Count=("Valid_RDS_Sample_Count", "sum"),
        Target_Condition_Transient_Count=("Target_Condition_Transient_Count", "sum"),
        Raw_Transient_Count=("Raw_Transient_Count", "sum"),
        Provisional_Min_Current_A=("Provisional_Min_Current_A", "first"),
    )
)

# Evidence row for the maximum observed RDS in each component.
max_idx = limit_runlevel.groupby("Test_ID")["Max_RDS_Instantaneous_Ohm"].idxmax()
max_evidence = limit_runlevel.loc[max_idx.dropna().astype(int), [
    "Test_ID", "Max_RDS_Transient_ID", "Max_RDS_Time_us"
]].rename(columns={
    "Max_RDS_Transient_ID": "Max_RDS_Evidence_Transient_ID",
    "Max_RDS_Time_us": "Max_RDS_Evidence_Time_us",
})
component_limit = component_limit.merge(max_evidence, on="Test_ID", how="left")

component_limit["Limit_Exceedance_Flag"] = np.where(
    component_limit["Limit_Exceedance_Runs"] > 0,
    "FLAGGED",
    np.where(
        component_limit["Runs_Evaluated"] > 0,
        "NOT FLAGGED",
        "NOT_EVALUATED",
    ),
)

component_limit["Limit_Exceedance_Margin_Ohm"] = (
    component_limit["Max_RDS_Instantaneous_Ohm"]
    - component_limit["Engineering_Limit_Ohm"]
)

component_limit["Limit_Scan_Status"] = np.where(
    component_limit["Runs_Evaluated"] > 0,
    np.where(component_limit["Runs_With_Error"] > 0, "PARTIAL_ERROR", "EVALUATED"),
    np.where(component_limit["Runs_With_Error"] > 0, "ERROR", "NOT_EVALUATED"),
)

display(component_limit)

print("\nModule C run-level status counts:")
print(limit_runlevel["Limit_Scan_Status"].value_counts(dropna=False))

print("\nModule C target-transient audit:")
print("Total run-level records:", len(limit_runlevel))
print("Expected target-condition transients:", int(limit_runlevel["Expected_Target_Transient_Count"].sum()))
print("Observed target-condition transients:", int(limit_runlevel["Target_Condition_Transient_Count"].sum()))
print("Valid RDS(t) samples:", int(limit_runlevel["Valid_RDS_Sample_Count"].sum()))

if (limit_runlevel["Limit_Scan_Status"] == "ERROR").any():
    print("\nModule C errors — DO NOT TRUST final limit flags until resolved:")
    display(
        limit_runlevel.loc[
            limit_runlevel["Limit_Scan_Status"] == "ERROR",
            [
                "Test_ID", "Run_ID", "Expected_Target_Transient_Count",
                "Target_Condition_Transient_Count", "Raw_Transient_Count",
                "PWM_State_Count", "Target_Condition_Match_Attempt_Count",
                "Limit_Scan_Reason"
            ],
        ]
    )


,Test_ID,Engineering_Limit_Ohm,Max_RDS_Instantaneous_Ohm,Limit_Exceedance_Count,Limit_Exceedance_Runs,Runs_Evaluated,Runs_With_Error,Valid_RDS_Sample_Count,Target_Condition_Transient_Count,Raw_Transient_Count,Provisional_Min_Current_A,Max_RDS_Evidence_Transient_ID,Max_RDS_Evidence_Time_us,Limit_Exceedance_Flag,Limit_Exceedance_Margin_Ohm,Limit_Scan_Status
0,6,8.0,1.909445,0,0,2,0,1365200,733,24382,0.15,1008,50.2,NOT FLAGGED,-6.090555,EVALUATED
1,8,8.0,0.802820,0,0,7,0,235600,589,64176,0.15,210,355.0,NOT FLAGGED,-7.197180,EVALUATED
2,9,8.0,0.766725,0,0,7,0,237200,593,57080,0.15,180,281.0,NOT FLAGGED,-7.233275,EVALUATED
3,10,8.0,30.119273,235410,7,7,0,236249,592,61996,0.15,48,222.0,FLAGGED,22.119273,EVALUATED
4,11,8.0,0.818582,0,0,7,0,219600,549,59447,0.15,196,405.0,NOT FLAGGED,-7.181418,EVALUATED
5,12,8.0,0.812568,0,0,7,0,264000,660,60349,0.15,211,270.0,NOT FLAGGED,-7.187432,EVALUATED
6,13,8.0,34.912634,19274,1,5,0,186474,470,11567,0.15,179,216.0,FLAGGED,26.912634,EVALUATED
7,14,8.0,0.858944,0,0,7,0,276000,690,50317,0.15,170,189.0,NOT FLAGGED,-7.141056,EVALUATED
8,20,8.0,0.811003,0,0,2,0,177200,443,40254,0.15,348,69.0,NOT FLAGGED,-7.188997,EVALUATED
9,23,8.0,0.886789,0,0,2,0,59600,149,19482,0.15,179,423.0,NOT FLAGGED,-7.113211,EVALUATED



Module C run-level status counts:
Limit_Scan_Status
EVALUATED    67
Name: count, dtype: int64

Module C target-transient audit:
Total run-level records: 67
Expected target-condition transients: 6900
Observed target-condition transients: 6900
Valid RDS(t) samples: 3656923


In [13]:
# ============================================================
# 13. MERGE MODULE A + MODULE B + MODULE C
# ============================================================

result = wide.merge(component_limit, on="Test_ID", how="left")

# Historical forecast evidence only — never used to create a new anomaly label.
result["Forecast_Residual"] = result["RDS100"] - result["Predicted_RDS100"]
result["Absolute_Forecast_Error"] = result["Forecast_Residual"].abs()
actual_abs = result["RDS100"].abs()
result["Relative_Error_Percent"] = np.where(
    actual_abs > 0,
    result["Absolute_Forecast_Error"] / actual_abs * 100.0,
    np.nan,
)
result["Module_B_Anomaly"] = pd.NA

RESULT_COLUMNS = [
    "Test_ID",
    "RDS0", "RDS33", "RDS66", "RDS100",
    "Delta_RDS_0_33", "Delta_RDS_33_66", "Delta_RDS_66_100",
    "ID33_ID0_Ratio",
    "Module_A_IF_Score_0", "Module_A_IF_Score_33", "Module_A_IF_Score_66", "Module_A_IF_Score_100",
    "Module_A_Novelty_Percentile_0", "Module_A_Novelty_Percentile_33", "Module_A_Novelty_Percentile_66", "Module_A_Novelty_Percentile_100",
    "Module_A_Flag_0", "Module_A_Flag_33", "Module_A_Flag_66", "Module_A_Flag_100",
    "Progressive_ML_Flag", "First_Anomaly_Stage",
    "Engineering_Limit_Ohm", "Provisional_Min_Current_A", "Max_RDS_Instantaneous_Ohm", "Limit_Exceedance_Count",
    "Limit_Exceedance_Runs", "Valid_RDS_Sample_Count", "Target_Condition_Transient_Count",
    "Limit_Exceedance_Margin_Ohm", "Limit_Exceedance_Flag", "Limit_Scan_Status",
    "Max_RDS_Evidence_Transient_ID", "Max_RDS_Evidence_Time_us",
    "Predicted_RDS100", "Forecast_Residual", "Absolute_Forecast_Error", "Relative_Error_Percent"
]

result = result[RESULT_COLUMNS].sort_values("Test_ID").reset_index(drop=True)
result.to_csv(RESULT_FILE, index=False)

print("Saved redesigned SPAD result:")
print(RESULT_FILE)
display(result)


Saved redesigned SPAD result:
C:\Users\Admin\Downloads\SPAD_V4_End_to_End_Inference.csv


,Test_ID,RDS0,RDS33,RDS66,RDS100,Delta_RDS_0_33,Delta_RDS_33_66,Delta_RDS_66_100,ID33_ID0_Ratio,Module_A_IF_Score_0,...,Target_Condition_Transient_Count,Limit_Exceedance_Margin_Ohm,Limit_Exceedance_Flag,Limit_Scan_Status,Max_RDS_Evidence_Transient_ID,Max_RDS_Evidence_Time_us,Predicted_RDS100,Forecast_Residual,Absolute_Forecast_Error,Relative_Error_Percent
0,6,0.483105,0.589925,0.696745,0.803565,0.106820,0.106820,0.106820,0.899883,-0.095767,...,733,-6.090555,NOT FLAGGED,EVALUATED,1008,50.2,0.691441,0.112124,0.112124,13.953360
1,8,0.486021,0.528422,0.630160,0.660393,0.042401,0.101739,0.030233,0.951254,-0.067558,...,589,-7.197180,NOT FLAGGED,EVALUATED,210,355.0,0.680075,-0.019681,0.019681,2.980269
2,9,0.512634,0.518506,0.629173,0.635507,0.005872,0.110667,0.006334,0.992978,0.061678,...,593,-7.233275,NOT FLAGGED,EVALUATED,180,281.0,0.628772,0.006734,0.006734,1.059705
3,10,13.334364,13.612318,13.235870,14.374252,0.277954,-0.376448,1.138382,0.908793,-0.030064,...,592,22.119273,FLAGGED,EVALUATED,48,222.0,0.691119,13.683133,13.683133,95.191964
4,11,0.518307,0.508018,0.609227,0.603363,-0.010290,0.101210,-0.005864,0.994881,-0.021844,...,549,-7.181418,NOT FLAGGED,EVALUATED,196,405.0,0.580966,0.022397,0.022397,3.712010
5,12,0.510677,0.510597,0.636908,0.610565,-0.000080,0.126311,-0.026342,1.005954,0.038035,...,660,-7.187432,NOT FLAGGED,EVALUATED,211,270.0,0.578886,0.031679,0.031679,5.188538
6,13,0.513423,0.544736,0.568940,24.675459,0.031312,0.024204,24.106520,0.964164,0.055620,...,470,26.912634,FLAGGED,EVALUATED,179,216.0,0.680075,23.995385,23.995385,97.243923
7,14,0.513944,0.542102,0.626834,0.672058,0.028157,0.084733,0.045224,0.973096,0.053862,...,690,-7.141056,NOT FLAGGED,EVALUATED,170,189.0,0.675940,-0.003882,0.003882,0.577635
8,20,0.572615,0.605515,0.638415,0.671315,0.032900,0.032900,0.032900,0.964323,-0.011346,...,443,-7.188997,NOT FLAGGED,EVALUATED,348,69.0,0.691119,-0.019804,0.019804,2.950053
9,23,0.563583,0.607752,0.651922,0.696091,0.044170,0.044170,0.044170,0.950154,-0.036992,...,149,-7.113211,NOT FLAGGED,EVALUATED,179,423.0,0.691119,0.004972,0.004972,0.714309


## 15. Optional Node/Express delivery

Python computes the SPAD result. Node/Express validates and normalizes the payload. MongoDB persistence remains outside this notebook and is not performed here.

`SEND_TO_BACKEND=False` is the production-safe default. Set it to `True` only after the actual project `SPAD_LOT_ID` and `/api/ai/results` endpoint are confirmed.

In [14]:

# ============================================================
# 14. FINAL SPAD V4 ASSESSMENT
# ============================================================

final_assessment = result.copy()

# Module A remains early-warning evidence only.
module_a_flag_columns = [
    "Module_A_Flag_0",
    "Module_A_Flag_33",
    "Module_A_Flag_66",
    "Module_A_Flag_100",
]

def _module_a_flag_present(row):
    for col in module_a_flag_columns:
        value = row.get(col, np.nan)
        if isinstance(value, str):
            if value.strip().upper() in {"FLAGGED", "TRUE", "1", "-1"}:
                return True
        elif isinstance(value, (bool, np.bool_)):
            if bool(value):
                return True
        elif pd.notna(value):
            try:
                if float(value) == -1:
                    return True
            except Exception:
                pass
    return False

final_assessment["Module_A_Early_Warning"] = (
    final_assessment.apply(_module_a_flag_present, axis=1)
    .map({True: "PRESENT", False: "NOT PRESENT"})
)

final_assessment["Module_B_Forecast_Status"] = np.where(
    final_assessment["Predicted_RDS100"].notna(),
    "AVAILABLE",
    "NOT_EVALUATED",
)

final_assessment["Engineering_Status"] = (
    final_assessment["Limit_Exceedance_Flag"]
    .map({
        "FLAGGED": "FLAGGED",
        "NOT FLAGGED": "NOT FLAGGED",
        "NOT_EVALUATED": "NOT_EVALUATED",
    })
    .fillna("NOT_EVALUATED")
)

# Engineering hard-limit compliance is the authoritative final status.
final_assessment["Overall_Status"] = final_assessment["Engineering_Status"]

# Integrity gates.
assert final_assessment["Test_ID"].is_unique, "Test_ID must be unique."
assert final_assessment["Predicted_RDS100"].notna().all(), "Missing Module B predictions."
assert final_assessment["Limit_Scan_Status"].eq("EVALUATED").all(), (
    "Module C must be fully evaluated before finalizing SPAD V4."
)
assert (final_assessment["Engineering_Limit_Ohm"] == ENGINEERING_LIMIT_OHM).all(), (
    "Engineering limit mismatch in final results."
)
assert (final_assessment["Provisional_Min_Current_A"] == PROVISIONAL_MIN_CURRENT_A).all(), (
    "Minimum-current configuration mismatch in final results."
)

display_columns = [
    "Test_ID",
    "RDS0", "RDS33", "RDS66", "RDS100",
    "Module_A_Novelty_Percentile_0",
    "Module_A_Novelty_Percentile_33",
    "Module_A_Novelty_Percentile_66",
    "Module_A_Novelty_Percentile_100",
    "First_Anomaly_Stage",
    "Module_A_Early_Warning",
    "Predicted_RDS100",
    "Forecast_Residual",
    "Absolute_Forecast_Error",
    "Relative_Error_Percent",
    "Engineering_Limit_Ohm",
    "Provisional_Min_Current_A",
    "Max_RDS_Instantaneous_Ohm",
    "Limit_Exceedance_Count",
    "Limit_Exceedance_Runs",
    "Limit_Exceedance_Margin_Ohm",
    "Max_RDS_Evidence_Transient_ID",
    "Max_RDS_Evidence_Time_us",
    "Limit_Exceedance_Flag",
    "Engineering_Status",
    "Overall_Status",
]

print("=" * 100)
print("SPAD V4 — FINAL ASSESSMENT")
print("=" * 100)
print("Components:", len(final_assessment))
print("Engineering limit:", f"{ENGINEERING_LIMIT_OHM:.6f} Ω")
print("Provisional minimum valid ID:", f"{PROVISIONAL_MIN_CURRENT_A:.3f} A")
print("Module C all evaluated:", bool(final_assessment["Limit_Scan_Status"].eq("EVALUATED").all()))
print("Module B all available:", bool(final_assessment["Predicted_RDS100"].notna().all()))
print("\nOverall status counts:")
print(final_assessment["Overall_Status"].value_counts(dropna=False))
print("\nModule A early-warning evidence:")
print(final_assessment["Module_A_Early_Warning"].value_counts(dropna=False))

display(
    final_assessment[display_columns]
    .sort_values("Test_ID")
    .reset_index(drop=True)
)

print("\nFLAGGED COMPONENTS")
display(
    final_assessment.loc[
        final_assessment["Overall_Status"].eq("FLAGGED"),
        display_columns
    ]
)

FINAL_ASSESSMENT_FILE = DOWNLOADS / "SPAD_V4_Final_Assessment1.csv"
final_assessment.to_csv(FINAL_ASSESSMENT_FILE, index=False)

print("\nSaved:", FINAL_ASSESSMENT_FILE)


SPAD V4 — FINAL ASSESSMENT
Components: 15
Engineering limit: 8.000000 Ω
Provisional minimum valid ID: 0.150 A
Module C all evaluated: True
Module B all available: True

Overall status counts:
Overall_Status
NOT FLAGGED    13
FLAGGED         2
Name: count, dtype: int64

Module A early-warning evidence:
Module_A_Early_Warning
PRESENT        10
NOT PRESENT     5
Name: count, dtype: int64


,Test_ID,RDS0,RDS33,RDS66,RDS100,Module_A_Novelty_Percentile_0,Module_A_Novelty_Percentile_33,Module_A_Novelty_Percentile_66,Module_A_Novelty_Percentile_100,First_Anomaly_Stage,...,Provisional_Min_Current_A,Max_RDS_Instantaneous_Ohm,Limit_Exceedance_Count,Limit_Exceedance_Runs,Limit_Exceedance_Margin_Ohm,Max_RDS_Evidence_Transient_ID,Max_RDS_Evidence_Time_us,Limit_Exceedance_Flag,Engineering_Status,Overall_Status
0,6,0.483105,0.589925,0.696745,0.803565,100.000000,100.000000,100.000000,100.000000,0.0,...,0.15,1.909445,0,0,-6.090555,1008,50.2,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
1,8,0.486021,0.528422,0.630160,0.660393,84.615385,84.615385,61.538462,38.461538,0.0,...,0.15,0.802820,0,0,-7.197180,210,355.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
2,9,0.512634,0.518506,0.629173,0.635507,7.692308,23.076923,7.692308,7.692308,NaN,...,0.15,0.766725,0,0,-7.233275,180,281.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
3,10,13.334364,13.612318,13.235870,14.374252,61.538462,92.307692,92.307692,92.307692,0.0,...,0.15,30.119273,235410,7,22.119273,48,222.0,FLAGGED,FLAGGED,FLAGGED
4,11,0.518307,0.508018,0.609227,0.603363,53.846154,69.230769,46.153846,53.846154,0.0,...,0.15,0.818582,0,0,-7.181418,196,405.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
5,12,0.510677,0.510597,0.636908,0.610565,23.076923,15.384615,53.846154,69.230769,NaN,...,0.15,0.812568,0,0,-7.187432,211,270.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
6,13,0.513423,0.544736,0.568940,24.675459,7.692308,46.153846,69.230769,92.307692,66.0,...,0.15,34.912634,19274,1,26.912634,179,216.0,FLAGGED,FLAGGED,FLAGGED
7,14,0.513944,0.542102,0.626834,0.672058,15.384615,46.153846,30.769231,30.769231,NaN,...,0.15,0.858944,0,0,-7.141056,170,189.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
8,20,0.572615,0.605515,0.638415,0.671315,46.153846,76.923077,69.230769,46.153846,0.0,...,0.15,0.811003,0,0,-7.188997,348,69.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED
9,23,0.563583,0.607752,0.651922,0.696091,69.230769,92.307692,84.615385,84.615385,0.0,...,0.15,0.886789,0,0,-7.113211,179,423.0,NOT FLAGGED,NOT FLAGGED,NOT FLAGGED



FLAGGED COMPONENTS


,Test_ID,RDS0,RDS33,RDS66,RDS100,Module_A_Novelty_Percentile_0,Module_A_Novelty_Percentile_33,Module_A_Novelty_Percentile_66,Module_A_Novelty_Percentile_100,First_Anomaly_Stage,...,Provisional_Min_Current_A,Max_RDS_Instantaneous_Ohm,Limit_Exceedance_Count,Limit_Exceedance_Runs,Limit_Exceedance_Margin_Ohm,Max_RDS_Evidence_Transient_ID,Max_RDS_Evidence_Time_us,Limit_Exceedance_Flag,Engineering_Status,Overall_Status
3,10,13.334364,13.612318,13.23587,14.374252,61.538462,92.307692,92.307692,92.307692,0.0,...,0.15,30.119273,235410,7,22.119273,48,222.0,FLAGGED,FLAGGED,FLAGGED
6,13,0.513423,0.544736,0.56894,24.675459,7.692308,46.153846,69.230769,92.307692,66.0,...,0.15,34.912634,19274,1,26.912634,179,216.0,FLAGGED,FLAGGED,FLAGGED



Saved: C:\Users\Admin\Downloads\SPAD_V4_Final_Assessment1.csv


In [ ]:
# ============================================================
# 15. PYTHON → NODE / EXPRESS DELIVERY
#
# Production delivery connects to the SPAD backend /api/ai/results
# MongoDB persistence remains controlled by the backend (dry-run).
# ============================================================

if "final_assessment" not in globals():
    raise RuntimeError(
        "`final_assessment` is not defined. "
        "Run the FINAL SPAD V4 ASSESSMENT cell immediately before this delivery cell."
    )

import json
import os
import requests

def build_backend_payload(df, lot_id):
    """Build the SPAD V4 payload without inventing IDs or flags."""
    required = [
        "Test_ID", "RDS0", "RDS33",
        "Module_A_IF_Score_33",
        "Module_A_Novelty_Percentile_33",
        "Predicted_RDS100",
        "Forecast_Residual",
        "Absolute_Forecast_Error",
        "Limit_Exceedance_Flag",
        "Engineering_Status",
        "Overall_Status",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing backend fields: {missing}")

    results = []

    for _, row in df.iterrows():
        component_id = f"TEST-{int(row['Test_ID']):02d}"

        # Method 2 explicit AI flag: use Module_A_Flag_33 (canonical early decision)
        m_a_flag = str(row.get("Module_A_Flag_33", "NOT_EVALUATED")).strip().upper()
        if m_a_flag not in {"FLAGGED", "NOT FLAGGED", "NOT_EVALUATED"}:
            m_a_flag = "NOT_EVALUATED"

        results.append({
            "Test_ID": component_id,
            "componentId": component_id,
            "RDS0": float(row["RDS0"]),
            "RDS33": float(row["RDS33"]),
            "Delta_RDS_0_33": float(row["Delta_RDS_0_33"]),

            # Module A — Isolation Forest (canonical early-trajectory 33% stage)
            "Module_A_IF_Score": float(
                row["Module_A_IF_Score_33"]
            ),
            "Module_A_Novelty_Percentile": float(
                row["Module_A_Novelty_Percentile_33"]
            ),
            "Module_A_Anomaly": m_a_flag,
            "Module_A_Flag": m_a_flag,

            # Preserve stage-wise values
            "Module_A_IF_Scores": {
                "0": float(row["Module_A_IF_Score_0"]),
                "33": float(row["Module_A_IF_Score_33"]),
                "66": float(row["Module_A_IF_Score_66"]),
                "100": float(row["Module_A_IF_Score_100"]),
            },
            "Module_A_Novelty_Percentiles": {
                "0": float(row["Module_A_Novelty_Percentile_0"]),
                "33": float(row["Module_A_Novelty_Percentile_33"]),
                "66": float(row["Module_A_Novelty_Percentile_66"]),
                "100": float(row["Module_A_Novelty_Percentile_100"]),
            },

            # Module B — Random Forest Future Prediction
            "Predicted_RDS100": float(row["Predicted_RDS100"]),
            "Forecast_Residual": float(row["Forecast_Residual"]),
            "Absolute_Forecast_Error": float(row["Absolute_Forecast_Error"]),
            "Relative_Error_Percent": float(row["Relative_Error_Percent"]),

            # Module C — Engineering Hard Limit Evidence
            "Engineering_Limit_Ohm": float(row["Engineering_Limit_Ohm"]),
            "Provisional_Min_Current_A": float(
                row["Provisional_Min_Current_A"]
            ),
            "Max_RDS_Instantaneous_Ohm": float(
                row["Max_RDS_Instantaneous_Ohm"]
            ),
            "Limit_Exceedance_Count": int(
                row["Limit_Exceedance_Count"]
            ),
            "Limit_Exceedance_Runs": int(
                row["Limit_Exceedance_Runs"]
            ),
            "Limit_Exceedance_Margin_Ohm": float(
                row["Limit_Exceedance_Margin_Ohm"]
            ),
            "Limit_Exceedance_Flag": str(
                row["Limit_Exceedance_Flag"]
            ),

            # Final engineering decision from notebook
            "Engineering_Status": str(
                row["Engineering_Status"]
            ),
            "Overall_Status": str(
                row["Overall_Status"]
            ),
        })

    return {
        "lotId": lot_id,
        "results": results
    }


if SEND_TO_BACKEND:
    if not LOT_ID:
        raise ValueError(
            "SPAD_LOT_ID is required when backend delivery is enabled."
        )

    backend_payload = build_backend_payload(final_assessment, LOT_ID)

    endpoint = (
        BACKEND_BASE_URL.rstrip("/")
        + "/api/ai/results"
    )

    print("=" * 90)
    print("SPAD V4 — BACKEND DELIVERY")
    print("=" * 90)
    print("Backend delivery enabled:", SEND_TO_BACKEND)
    print("MongoDB write from notebook: NONE")
    print("lotId:", backend_payload["lotId"])
    print("Results:", len(backend_payload["results"]))
    print("POST:", endpoint)

    try:
        response = requests.post(
            endpoint,
            json=backend_payload,
            timeout=15
        )

    except requests.exceptions.ConnectionError as exc:
        raise RuntimeError(
            f"Backend unavailable at {endpoint}."
        ) from exc

    except requests.exceptions.Timeout as exc:
        raise RuntimeError(
            f"Backend request timed out: {endpoint}."
        ) from exc

    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            f"Backend request failed: {exc}"
        ) from exc

    print("HTTP status:", response.status_code)

    try:
        response_body = response.json()
        print(
            json.dumps(
                response_body,
                indent=2
            )
        )
    except Exception:
        response_body = response.text
        print(response.text)

    if not response.ok:
        raise RuntimeError(
            f"Backend rejected SPAD results: "
            f"HTTP {response.status_code}"
        )
else:
    print("Backend delivery disabled.")
    print("No HTTP request was sent.")

## 16. Final production architecture

```text
Raw NASA MAT
   ↓
Frozen ON-window / scalar RDS extraction
   ↓
0/33/66/100% trajectory
   ├── Module A: progressive novelty evidence
   ├── Module B: frozen RDS100 forecast
   └── Module C: engineer hard-limit scan over valid RDS(t)
                         ↓
                   Final engineering status
                         ↓
                Optional Node/Express POST
```

Module A is retained as early-warning evidence, not an unconditional production PASS/FAIL classifier. Module B is predictive evidence. Module C is the authoritative engineering hard-limit decision.

Production configuration:
- Engineering RDS hard limit: 8.0 Ω
- Provisional minimum valid current for instantaneous RDS(t): 0.15 A
- Backend delivery: disabled by default
- MongoDB writes: not performed by this notebook
- No FFT/spectral branch
- No diagnostic/ground-truth test cells